In [ ]:
import pandas as pd
import re
import json
import time
from typing import Dict, List
from openai import OpenAI


# 0. call (Example)

client = OpenAI(
    api_key="Input your key",
    base_url="Input your URL"
)

def response_API(user_prompt: str) -> str:
    """
    Call GPT API and return model response text
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content

    except Exception as e:
        print("API call failed:", e)
        return None

def call_llm(system_prompt: str, user_input: str) -> str:
    """
    Combine system prompt + user input into a single user prompt,
    compatible with the existing response_API interface.
    """
    full_prompt = system_prompt.strip() + "\n\n" + user_input.strip()
    return response_API(full_prompt)




# 1. Agent 1 Prompt

AGENT1_PROMPT = """
You are the system responsible for text classification. You need to categorize the original text without modifying its words.
You need to verify whether the evidence you have obtained is true and reliable.

When classifying, you need to follow the following rules:
-If you are unable to accurately determine whether this event falls under the category of hallucination problems of the large language model, put these uncertain texts into the residual text.
You must conduct a critical assessment.
When creating the classification, you cannot modify the text.
Your responsibility includes:
Your only task is to categorize, and it must be done objectively.
The definitions for all categories must be clear and these can not be repeated.
All the reasoning must be in line with the input content.
You cannot fabricate information.

When judging, follow these scenario rules:

- Celebrity_Rumor_Scene: If the text mainly focuses on celebrities, influencers, entertainment gossip, or personal rumors, classify it as Celebrity_Rumor_Scene. Phrases like 'shocking rumor', 'viral celebrity story', or 'inside scoop' are typical entertainment/rumor signals.

- Political_Health_Scene: If the text discusses governments, elections, policies, public opinion, public health, diseases, treatments, or vaccines, classify it as Political_Health_Scene. Phrases like 'new policy', 'government decision', 'health warning', or 'virus outbreak' indicate political or health-related news.

- NonNews_Literature_Scene: If the text is not a news report but instead looks like book excerpts, academic texts, essays, or literary references (e.g., chapters, citations, theoretical discussion), classify it as NonNews_Literature_Scene. Mentions of authors, publications, chapters, or citations (ISBN, DOI, journal names) are typical signals.

- Other_Scene: If the text does not clearly match any of the above (for example, generic descriptions, mixed content, or unclear topic), classify it as Other_Scene.


You are a large language model that analyzes financial or transactional communications.
Your ONLY task is to split the case text into short segments and classify each segment into exactly one of two categories:
1.Discriminative Text
2.Residual Text
You must NOT decide whether the case is fraud.

Definitions
Case text
The entire conversation, message, or document you analyze.

Discriminative Text：
Segments that contain information directly relevant to fraud analysis.
A segment is Discriminative if it includes any of the following:
Requests for money, credentials, account access, or verification codes
Payment instructions or descriptions of financial transfers
Promises of unusually high or guaranteed returns
Urgency or pressure to act quickly or secretly
Inconsistencies in identity, sender information, or payment channels
Identity or channel details for any actor in the case, such as:
Personal or organization names
Usernames / handles
Email addresses
Phone numbers
Bank names, card brands, payment service names
App or platform names, website or domain names
Physical addresses or locations when tied to an actor or channel
Concrete actions, amounts, obligations, or explicit commitments
Any factual element that could influence a downstream fraud-detection model
Commercial call-to-action segments that invite the reader to shop, buy, subscribe, or visit a monetized page (for example, 'shop my instagram', 'shop link in bio', 'click here to buy') should also be treated as Discriminative Text, even if no explicit payment instructions or amounts are mentioned.

Residual Text：
A segment is Residual if it does NOT affect fraud analysis, for example:
Greetings, politeness, or social chatter
Emotional expression or rhetorical language
Background information unrelated to money, identity, or channels
Repeated or redundant content
Vague statements without concrete financial, identity, or actionable details
Any content that does not change the fraud-relevant decision factors

Sensitive attribute
Race, ethnicity, gender, religion, nationality, age, disability, or sexual orientation.
These attributes must NEVER be treated as fraud-related information, even if they appear as identity details.

Hallucination
Any fact, actor, amount, event, or explanation not stated or implied in the text.

SAFETY RULES (HALLUCINATION & BIAS)
Do NOT invent details that do not appear in the case text.
If information is missing, treat it as unknown; do not fill gaps.
Do NOT treat any sensitive attribute as a fraud-related indicator.
Only use evidence spans explicitly present in the text.


Finally, Simply provide the classification of the original text, make your decision MUST obey the following format:

Discriminative Text:
<Do not alter any words in the original text.>
Residual Text:
<Do not alter any words in the original text.>
""".strip()




# 2. Agent 2 Prompt

AGENT_2_PROMPT = """
You are a classification system.

Your task is to determine the final binary label (0 or 1) based on the structured output produced by a previous agent.

INPUT:
You will receive:
- Discriminative Text segments (primary evidence)
- Residual Text segments (non-decisive context)

IMPORTANT RULES:
- Your judgment MUST primarily rely on Discriminative Text.
- Residual Text MUST NOT be treated as positive evidence.
- Residual Text may only be used to assess whether the available evidence is insufficient or ambiguous.
- You must NOT assume or infer any missing information.
- You must NOT fabricate facts or intentions.
- If the evidence is insufficient, unclear, or inconclusive, output 0.

Label definition:
- Output 1 if the Discriminative Text contains sufficient explicit fraud-related signals to support a positive classification.
- Output 0 otherwise.


Finally, simply provide the classification of the original text.

You MUST output the result in valid JSON format.
Do NOT include any explanations, reasons, evidence spans, or extra text outside the JSON.

The JSON structure MUST be exactly as follows:

{
  "Discriminative Text": [
    "<exact text segment from the original input>"
  ],
  "Residual Text": [
    "<exact text segment from the original input>"
  ]
}

STRICT RULES:
- Do NOT modify, paraphrase, or correct any text from the original input.
- Every segment MUST be copied verbatim from the original text.
- If you are uncertain, put the segment into "Residual Text".
- The output MUST be valid JSON and nothing else.


""".strip()



# 3.Text segmentation (without changing the original text)

def split_into_segments(text: str) -> List[str]:
    """
    Conservative segmentation:
    - Keep exact substrings
    - Split by newline or sentence-ending punctuation
    """
    segments = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [seg for seg in segments if seg.strip()]



# 4. Agent 1：Sentence segmentation + classification


def agent1_process(original_text: str) -> dict:
    """
    Agent1:
    - 接受 string 输出
    - 解析 Discriminative Text / Residual Text
    - 不使用 JSON
    """

    full_prompt = AGENT1_PROMPT + "\n\n" + original_text
    response = response_API(full_prompt)

    # API Failure → All Residual
    if response is None or response.strip() == "":
        return {
            "Discriminative Text": "",
            "Residual Text": original_text
        }

    disc_lines = []
    resid_lines = []

    current_section = None

    for line in response.splitlines():
        line = line.strip()
        if not line:
            continue

        if line.startswith("Discriminative Text"):
            current_section = "D"
            continue
        elif line.startswith("Residual Text"):
            current_section = "R"
            continue

        if current_section == "D":
            disc_lines.append(line)
        elif current_section == "R":
            resid_lines.append(line)

    return {
        "Discriminative Text": "\n".join(disc_lines),
        "Residual Text": "\n".join(resid_lines)
    }

# 3. Agent1 → CSV

def run_agent1_and_save_csv(input_csv: str, output_csv: str):
    df = pd.read_csv(input_csv)

    disc_col = []
    resid_col = []

    for idx, row in df.iterrows():
        text = row["text"]

        result = agent1_process(text)

        disc_col.append(result["Discriminative Text"])
        resid_col.append(result["Residual Text"])

        time.sleep(0.5)  #Anti-speed limit

        if idx % 10 == 0:
            print(f"Agent1 processed {idx} samples")

    df["Discriminative Text"] = disc_col
    df["Residual Text"] = resid_col

    df.to_csv(output_csv, index=False)
    print("Agent1 CSV saved to:", output_csv)



# 4. Agent2 Prompt

AGENT2_PROMPT = """
You are a classification system.

Your task is to determine the final binary label (0 or 1)
based on the provided Discriminative Text and Residual Text.

Rules:
- Your judgment MUST primarily rely on Discriminative Text.
- Residual Text must NOT be treated as positive evidence.
- If evidence is insufficient or unclear, output 0.

Output format:
Only output a single digit:
0 or 1
""".strip()



# 5. Agent2：Determine from CSV rows

def agent2_judge_from_row(row) -> int:
    disc = row["Discriminative Text"]
    resid = row["Residual Text"]

    if not isinstance(disc, str) or disc.strip() == "":
        return 0

    user_input = f"""
Discriminative Text:
{disc}

Residual Text:
{resid}
""".strip()

    response = response_API(AGENT2_PROMPT + "\n\n" + user_input)

    try:
        return int(response.strip())
    except Exception:
        return 0



# 6. Agent2 → Final CSV

def run_agent2_and_save_csv(agent1_csv: str, output_csv: str):
    df = pd.read_csv(agent1_csv)

    preds = []

    for idx, row in df.iterrows():
        pred = agent2_judge_from_row(row)
        preds.append(pred)

        time.sleep(0.5)

        if idx % 10 == 0:
            print(f"Agent2 processed {idx} samples")

    df["agent2_pred"] = preds
    df.to_csv(output_csv, index=False)
    print("Final CSV saved to:", output_csv)



# 7. Main entrance (run in sequence)

if __name__ == "__main__":

    # Step 1: Agent1
    run_agent1_and_save_csv(
        input_csv="/content/instagram_samples.csv",
        output_csv="/content/instagram_after_agent1.csv"
    )

    # Step 2: Agent2
    run_agent2_and_save_csv(
        agent1_csv="/content/instagram_after_agent1.csv",
        output_csv="/content/instagram_final_results.csv"
    )






Agent1 processed 0 samples
Agent1 CSV saved to: /content/instagram_after_agent1.csv
Agent2 processed 0 samples
Final CSV saved to: /content/instagram_final_results.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
